In [0]:
catalog = "sandbox"
schema = "silver"
df_equipos_silver = spark.table(f"{catalog}.{schema}.equipos")
df_ordenes_silver = spark.table(f"{catalog}.{schema}.ordenes")

In [0]:
from pyspark.sql.functions import datediff, col, when

"""df_ordenes_enriched = df_ordenes_silver.withColumn(
    "tiempo_resolucion",
    datediff(col("fecha_cierre"), col("fecha_apertura"))
)

df_ordenes_enriched = df_ordenes_enriched.withColumn(
    "orden_abierta",
    col("fecha_cierre").isNull()
)

df_ordenes_enriched = df_ordenes_enriched.withColumn(
    "costo_categorizado",
    when(col("costo_estimado") < 1000, "bajo")
    .when(col("costo_estimado") <= 1000, "medio")
    .otherwise("alto")
)"""

df_ordenes_enriched = df_ordenes_silver.selectExpr(
    "*", # todas las columnas originales, conviene ser explicito
    "DATEDIFF(fecha_cierre, fecha_apertura) as tiempo_resolucion",
    "CASE WHEN fecha_cierre = '1900-01-01T00:00:00' THEN true ELSE false END as orden_abierta",
    "CASE WHEN costo_estimado < 10000 THEN 'bajo' WHEN costo_estimado >= 10000 AND costo_estimado < 50000 THEN 'medio' ELSE 'alto' END as costo_categorizado"
)

In [0]:
# join 
df_ordenes_enriched = df_ordenes_enriched.join(
    df_equipos_silver.select("equipo_id", "fabricante", "region", "tipo", "estado"), 
    on="equipo_id", 
    how="inner")

In [0]:
df_ordenes_enriched.createOrReplaceTempView("ordenes_enriched_vw")

In [0]:
%sql
MERGE INTO sandbox.gold.ordenes_enriched AS tgt
USING ordenes_enriched_vw AS src
ON tgt.orden_id = src.orden_id
WHEN NOT MATCHED THEN INSERT *